<h1 face="Times New Roma" color="#3f134f">Foundations of Probability</h1>

<p>
  <a href="https://github.com/daodavid">author: daodeiv (David Stankov)</a>
</p>


<h6>
  <font size="4" face="Times New Roma" color="#3f134f">
    <ul style="margin-left: 30px">
      <li><a href="#abstract">Abstract</a></li><br>
      <li><a href="#what">What is probability?</a></li><br>
      <li><a href="#experiment">Random experiments, outcomes, and the sample space</a></li><br>
      <li><a href="#events">Events</a></li><br>
      <li><a href="#axioms">Axioms of probability</a></li><br>
      <li><a href="#empirical">Theoretical vs empirical probability</a></li><br>
      <li><a href="#ref">References</a></li><br>
    </ul>
  </font>
</h6>


<h2 id="abstract" face="Times New Roma" color="#270336">Abstract</h2>

This notebook introduces the language of probability used throughout machine learning: uncertainty as a number in $[0,1]$, random experiments and sample spaces, events, and Kolmogorov's axioms. A fair coin is the running example. We first write the **theoretical** probabilities, then simulate many flips with NumPy to see the **empirical** relative frequency approach those values. Later notebooks (algebra of events, conditional probability, Bayes) build on this foundation.


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Same seed => same simulated results every run
rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")


<h2 id="what" face="Times New Roma" color="#270336">1. What is probability?</h2>

Probability is the numerical language of uncertainty. It assigns a number between $0$ and $1$ to an event:

* $0$ means the event is impossible
* $1$ means the event is certain
* values in between describe degrees of likelihood

A fair coin has two equally likely outcomes, heads ($H$) and tails ($T$). Therefore

$$
P(H)=\frac{1}{2}=0.5, \qquad P(T)=\frac{1}{2}=0.5
$$

These are **theoretical** probabilities: they come from the model (fair coin), not from data.


In [ ]:
outcomes = ["Heads", "Tails"]
probabilities = [0.5, 0.5]

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(outcomes, probabilities, color=["#3f134f", "#6b4c7a"])
ax.set_ylim(0, 1)
ax.set_ylabel("Probability")
ax.set_title("A fair coin: theoretical probabilities")
for i, p in enumerate(probabilities):
    ax.text(i, p + 0.03, f"{p:.1f}", ha="center")
plt.show()


<h2 id="experiment" face="Times New Roma" color="#270336">2. Random experiments, outcomes, and the sample space</h2>

A **random experiment** is a process whose exact result is uncertain, even though the possible results are known in advance (flip a coin, roll a die, train a classifier on a shuffled split).

* An **outcome** is one possible result of a single run, written $\omega$.
* The **sample space** $\Omega$ is the set of all outcomes.

For one fair coin flip:

$$
\Omega = \{H, T\}
$$

For two flips the sample space is the set of ordered pairs:

$$
\Omega = \{HH, HT, TH, TT\}
$$

Each elementary outcome of a fair coin (or a fair pair of flips) is equally likely, so

$$
P(\omega) = \frac{1}{|\Omega|}
\quad \text{for every } \omega \in \Omega
$$


In [ ]:
# Sample space of two independent fair flips
omega = [a + b for a in "HT" for b in "HT"]
p = 1 / len(omega)

df = pd.DataFrame({"outcome": omega, "P(outcome)": [p] * len(omega)})
display(df)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(omega, [p] * len(omega), color="#3f134f")
ax.set_ylim(0, 0.5)
ax.set_ylabel("Probability")
ax.set_title(r"Two fair flips: $|\Omega|=4$, each outcome has probability $1/4$")
plt.show()


<h2 id="events" face="Times New Roma" color="#270336">3. Events</h2>

An **event** $A$ is a subset of the sample space: $A \subseteq \Omega$. We say the event occurs if the observed outcome lies in $A$.

Examples with two flips, $\Omega=\{HH, HT, TH, TT\}$:

* first coin is heads: $A=\{HH, HT\}$
* both coins match: $B=\{HH, TT\}$
* at least one heads: $C=\{HH, HT, TH\}$

If every outcome is equally likely,

$$
P(A) = \frac{|A|}{|\Omega|}
$$

so $P(A)=2/4=1/2$, $P(B)=1/2$, and $P(C)=3/4$.

Useful combinations:

* complement: $A^{c} = \Omega \setminus A$, and $P(A^{c})=1-P(A)$
* union $A \cup B$: at least one of the two events
* intersection $A \cap B$: both events


In [ ]:
omega = {"HH", "HT", "TH", "TT"}
A = {"HH", "HT"}           # first is heads
B = {"HH", "TT"}           # both match
C = {"HH", "HT", "TH"}     # at least one heads

def P(event):
    return len(event) / len(omega)

print(f"|Ω| = {len(omega)}")
print(f"P(A) first heads     = {P(A):.2f}")
print(f"P(B) both match      = {P(B):.2f}")
print(f"P(C) at least 1 head = {P(C):.2f}")
print(f"P(A ∩ B)             = {P(A & B):.2f}")
print(f"P(A ∪ B)             = {P(A | B):.2f}")
print(f"P(A^c)               = {P(omega - A):.2f}")


<h2 id="axioms" face="Times New Roma" color="#270336">4. Axioms of probability</h2>

Kolmogorov's axioms are the rules every probability assignment must satisfy. For events $A,B \subseteq \Omega$:

$$
\begin{aligned}
P(A) &\ge 0 \\
P(\Omega) &= 1 \\
P(A \cup B) &= P(A) + P(B) \quad \text{whenever } A \cap B = \emptyset
\end{aligned}
$$

The third axiom extends to a countable sequence of pairwise disjoint events. Immediate consequences:

$$
P(\emptyset)=0, \qquad P(A^{c})=1-P(A)
$$

and the inclusion-exclusion formula (always true, disjoint or not)

$$
P(A \cup B) = P(A) + P(B) - P(A \cap B)
$$

Check it on the two-flip events above: $P(A)+P(B)-P(A\cap B)=1/2+1/2-1/4=3/4=P(A\cup B)$.


<h2 id="empirical" face="Times New Roma" color="#270336">5. Theoretical vs empirical probability</h2>

The theoretical value $P(H)=1/2$ is a statement about the model. If we actually flip a coin $n$ times and let $N_H$ be the number of heads, the **empirical** (relative-frequency) estimate is

$$
\hat{P}_n(H) = \frac{N_H}{n}
$$

The law of large numbers says $\hat{P}_n(H) \to P(H)$ as $n \to \infty$. Below we simulate that convergence.


In [ ]:
n_max = 2000
flips = rng.choice(["H", "T"], size=n_max)
is_heads = (flips == "H").astype(float)
empirical = np.cumsum(is_heads) / np.arange(1, n_max + 1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(empirical, color="#3f134f", lw=1.5, label=r"$\hat{P}_n(H)$")
ax.axhline(0.5, color="gray", ls="--", label=r"theoretical $P(H)=1/2$")
ax.set_xlabel("number of flips $n$")
ax.set_ylabel("relative frequency of heads")
ax.set_title("Empirical probability approaches the theoretical value")
ax.legend()
ax.set_ylim(0, 1)
plt.show()

print(f"after {n_max} flips: empirical P(H) = {empirical[-1]:.3f}")


<h2 id="ref" face="Times New Roma" color="#270336">References</h2>

1. Kolmogorov, A. N. *Foundations of the Theory of Probability*.
2. Grimmett, G. and Stirzaker, D. *Probability and Random Processes*.
3. The companion notebook in this folder, *Probability basics*, continues with the algebra of events, conditional probability, and Bayes' theorem.
